# Deep Learning Lab: Job-Scam Detection and Threshold Tuning

Companion notebook for *AI for Security and Security for AI*.


## Goal

This notebook shows a deployment-oriented workflow for security text classification:

1. train a baseline classifier on the DiFrauD Job Scams data,
2. tune the decision threshold on validation data,
3. report precision, recall, F1, PR-AUC, and the confusion matrix,
4. optionally train a small neural text model when TensorFlow is available.

The central lesson is that a model outputs a score; the security team still needs an operational threshold.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Dataset

The notebook downloads the `job_scams` split from DiFrauD. DiFrauD stores examples as JSONL files with two fields: `text` and `label`. Here, `label=1` means deceptive/fraudulent and `label=0` means non-deceptive.


In [ ]:
DIFRAUD_BASE = "https://huggingface.co/datasets/difraud/difraud/resolve/main"

def load_difraud_domain(domain, splits=("train", "validation", "test")):
    frames = []
    for split in splits:
        url = f"{DIFRAUD_BASE}/{domain}/{split}.jsonl"
        tmp = pd.read_json(url, lines=True)
        tmp["split"] = split
        tmp["domain"] = domain
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df

df = load_difraud_domain("job_scams")
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()
print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["label"].value_counts())
train_df.head()


## Baseline: TF-IDF + Logistic Regression


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")),
])
clf.fit(train_df["text"], train_df["label"])
val_scores = clf.predict_proba(val_df["text"])[:, 1]
test_scores = clf.predict_proba(test_df["text"])[:, 1]


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, average_precision_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay
)

def binary_metrics(y_true, scores, threshold=0.5):
    y_pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"threshold": threshold, "accuracy": accuracy_score(y_true, y_pred),
           "precision": p, "recall": r, "f1": f1,
           "pr_auc": average_precision_score(y_true, scores)}
    try:
        out["roc_auc"] = roc_auc_score(y_true, scores)
    except Exception:
        out["roc_auc"] = np.nan
    return out

def choose_threshold(y_true, scores, beta=2.0, min_precision=None):
    best = None
    for th in np.linspace(0.01, 0.99, 99):
        y_pred = (scores >= th).astype(int)
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        if min_precision is not None and p < min_precision:
            continue
        beta2 = beta * beta
        fbeta = (1 + beta2) * p * r / (beta2 * p + r + 1e-12)
        if best is None or fbeta > best["fbeta"]:
            best = {"threshold": float(th), "precision": float(p), "recall": float(r), "fbeta": float(fbeta)}
    return best

default_metrics = binary_metrics(test_df["label"].values, test_scores, threshold=0.5)
choice = choose_threshold(val_df["label"].values, val_scores, beta=2.0, min_precision=0.20)
tuned_metrics = binary_metrics(test_df["label"].values, test_scores, threshold=choice["threshold"])
pd.DataFrame([default_metrics, tuned_metrics], index=["default threshold", "validation-tuned threshold"]).round(4)


In [ ]:
threshold = choice["threshold"]
y_pred = (test_scores >= threshold).astype(int)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(test_df["label"], y_pred, ax=ax, values_format="d")
ax.set_title(f"Confusion matrix at threshold={threshold:.2f}")
plt.show()
fig, ax = plt.subplots(figsize=(5, 4))
PrecisionRecallDisplay.from_predictions(test_df["label"], test_scores, ax=ax)
ax.set_title("Precision-recall curve")
plt.show()


## Optional: small neural text model

Run this section when TensorFlow is available. Keep it small for a book companion notebook.


In [ ]:
RUN_TENSORFLOW_MODEL = False  # Change to True if TensorFlow is installed.

if RUN_TENSORFLOW_MODEL:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    max_tokens = 30000
    seq_len = 250
    vectorizer = layers.TextVectorization(max_tokens=max_tokens, output_mode="int", output_sequence_length=seq_len)
    vectorizer.adapt(train_df["text"].values)
    model = models.Sequential([
        vectorizer,
        layers.Embedding(max_tokens, 128),
        layers.Conv1D(128, 5, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(train_df["text"].values, train_df["label"].values,
              validation_data=(val_df["text"].values, val_df["label"].values),
              epochs=5, batch_size=64, verbose=2)
    val_scores_nn = model.predict(val_df["text"].values, verbose=0).ravel()
    test_scores_nn = model.predict(test_df["text"].values, verbose=0).ravel()
    choice_nn = choose_threshold(val_df["label"].values, val_scores_nn, beta=2.0, min_precision=0.20)
    display(pd.DataFrame([
        binary_metrics(test_df["label"].values, test_scores_nn, threshold=0.5),
        binary_metrics(test_df["label"].values, test_scores_nn, threshold=choice_nn["threshold"])
    ], index=["NN default threshold", "NN validation-tuned threshold"]).round(4))
else:
    print("TensorFlow section skipped. Set RUN_TENSORFLOW_MODEL=True to run it.")


## Practitioner takeaway

For imbalanced security tasks, the default threshold of 0.5 is rarely the final deployment decision. Threshold tuning should reflect operational costs.
